# PYTORCH MASTERY — NOTEBOOK 2
## Chapters 6–10: Indexing and Slicing → Reshaping → Broadcasting → Mathematical Operations → Matrix Operations

Continuing directly from Notebook 1.


### PYTORCH MASTERY PROGRESS
Current Chapter: 10 | Current Level: Tensor Operations

- [x] Chapters 1–5 *(Notebook 1)*
- [x] Indexing and Slicing
- [x] Reshaping
- [x] Broadcasting
- [x] Mathematical Operations
- [x] Matrix Operations


---
# CHAPTER 6 — INDEXING AND SLICING

## 1. Concept
Tensor indexing extends NumPy's indexing rules directly — basic slicing, boolean masking, and "fancy" (array-based) indexing all work identically in spirit, with the crucial addition that **basic slicing returns a view (shares memory)** while **boolean/fancy indexing returns a copy** — a distinction that matters enormously once you start modifying indexed results.

## 2. Basic Indexing and Slicing


In [ ]:
import torch

x = torch.arange(24).reshape(4, 6)
print("x:\n", x)

print("\nx[0]:            ", x[0])              # first row
print("x[0, 0]:          ", x[0, 0].item())        # single element (as a Python scalar via .item())
print("x[:, 0]:          ", x[:, 0])              # first column (every row)
print("x[1:3, 2:5]:\n", x[1:3, 2:5])              # a sub-rectangle: rows 1-2, columns 2-4
print("\nx[-1]:            ", x[-1])              # last row (negative indexing, exactly like Python lists)
print("x[:, ::2]:\n", x[:, ::2])                   # every other column (step slicing)


x:
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17],
        [18, 19, 20, 21, 22, 23]])

x[0]:             tensor([0, 1, 2, 3, 4, 5])
x[0, 0]:           0
x[:, 0]:           tensor([ 0,  6, 12, 18])
x[1:3, 2:5]:
 tensor([[ 8,  9, 10],
        [14, 15, 16]])

x[-1]:             tensor([18, 19, 20, 21, 22, 23])
x[:, ::2]:
 tensor([[ 0,  2,  4],
        [ 6,  8, 10],
        [12, 14, 16],
        [18, 20, 22]])


### Line-by-Line Explanation
- `x[0, 0]` returns a **0-dimensional tensor** (a scalar tensor, not a Python number) — `.item()` extracts it as a plain Python `int`/`float`. This distinction matters: a 0-d tensor still participates in autograd/GPU placement, while `.item()`'s result is a plain Python number, completely detached from any tensor machinery — calling `.item()` inside a training loop repeatedly is also a well-known, subtle performance issue (Chapter 74), since it forces a GPU→CPU synchronization every time.
- `x[1:3, 2:5]` — standard Python slice syntax extends directly to multiple dimensions, comma-separated — each dimension gets its own independent slice.
- Basic slices like these **share memory** with the original tensor — this is important enough to demonstrate explicitly next.

## 3. Views vs. Copies — the Critical Distinction


In [ ]:
x = torch.arange(12).reshape(3, 4)
print("Original x:\n", x)

# Basic slicing returns a VIEW -- modifying it modifies the ORIGINAL tensor too
row_view = x[0]
row_view[0] = 999
print("\nAfter modifying row_view[0] = 999, ORIGINAL x is ALSO changed:\n", x)

# Boolean indexing returns a COPY -- modifying it does NOT affect the original
x2 = torch.arange(12).reshape(3, 4)
mask_result = x2[x2 > 5]
mask_result[0] = -1
print("\nOriginal x2 UNCHANGED after modifying the boolean-indexed result:\n", x2)
print("mask_result (independent copy):", mask_result)


Original x:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

After modifying row_view[0] = 999, ORIGINAL x is ALSO changed:
 tensor([[999,   1,   2,   3],
        [  4,   5,   6,   7],
        [  8,   9,  10,  11]])

Original x2 UNCHANGED after modifying the boolean-indexed result:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
mask_result (independent copy): tensor([-1,  7,  8,  9, 10, 11])


### Internal Working
Basic slicing (`x[0]`, `x[1:3]`, `x[:, 0]`) computes a new tensor that points to the **same underlying storage** as the original, just with different offset/stride metadata (Chapter 7 explains strides fully) — no data is copied, which is why it's extremely fast but also why modifications propagate back to the original. Boolean masking and "fancy" (integer-array) indexing genuinely cannot be expressed as a simple offset/stride view (the selected elements aren't contiguous or regularly spaced in general), so PyTorch **must** allocate new memory and copy the selected values — producing an independent tensor.

## 4. Boolean Masking — the Standard ML Filtering Pattern


In [ ]:
scores = torch.tensor([0.2, 0.9, 0.4, 0.95, 0.1, 0.7])

mask = scores > 0.5
print("mask:", mask)                    # a boolean tensor, same shape as scores
print("scores[mask]:", scores[mask])     # only the values where mask is True

# A genuinely common ML pattern: zero out (or replace) values that don't meet a condition
scores_thresholded = scores.clone()      # .clone() -- an explicit COPY, so we don't mutate the original
scores_thresholded[scores_thresholded <= 0.5] = 0.0
print("\nThresholded (below 0.5 -> 0):", scores_thresholded)


mask: tensor([False,  True, False,  True, False,  True])
scores[mask]: tensor([0.9000, 0.9500, 0.7000])

Thresholded (below 0.5 -> 0): tensor([0.0000, 0.9000, 0.0000, 0.9500, 0.0000, 0.7000])


### `.clone()` vs. Slicing — Making an Explicit, Intentional Copy
`.clone()` is PyTorch's explicit "give me a genuine, independent copy of this tensor" method — the direct analogue of NumPy's `.copy()`. Use it deliberately whenever you're about to modify a tensor in-place but need to preserve the original's values elsewhere — a common, important defensive habit (echoing the Scikit-learn course's `X.copy()` convention inside custom transformers).

### Fancy (Integer Array) Indexing


In [ ]:
x = torch.arange(20).reshape(4, 5)
print("x:\n", x)

indices = torch.tensor([0, 2, 3])
print("\nx[indices] -- select specific ROWS by index:\n", x[indices])

# A genuinely common deep learning use: selecting specific batch elements, e.g. the
# hardest examples by loss value, or gathering embeddings for specific token IDs
row_idx = torch.tensor([0, 1, 2])
col_idx = torch.tensor([1, 2, 3])
print("\nx[row_idx, col_idx] -- select SPECIFIC (row, col) PAIRS, not a rectangle:", x[row_idx, col_idx])


x:
 tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])

x[indices] -- select specific ROWS by index:
 tensor([[ 0,  1,  2,  3,  4],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])

x[row_idx, col_idx] -- select SPECIFIC (row, col) PAIRS, not a rectangle: tensor([ 1,  7, 13])


### Line-by-Line Explanation
- `x[row_idx, col_idx]` with two equal-length index tensors selects `(row_idx[0], col_idx[0])`, `(row_idx[1], col_idx[1])`, `(row_idx[2], col_idx[2])` — i.e., **paired** coordinates, not the Cartesian-product rectangle a naive reading might suggest. This exact pattern is how you'd extract, e.g., "the predicted probability of the TRUE class for each sample in a batch" — `predictions[torch.arange(batch_size), true_labels]`, an idiom you will see constantly in loss-function implementations.

### Common Mistakes
- Modifying a basic-slice view and being surprised the original tensor changed — always `.clone()` first if independence is needed.
- Assuming boolean-indexed results share memory with the original (they don't) — attempting to modify the original *through* a boolean-indexed result silently fails to propagate.
- Calling `.item()` repeatedly inside a hot training loop — each call forces a synchronization point (especially costly on GPU), a genuine, measurable performance mistake (Chapter 74).
- Confusing `x[[0,1], [0,1]]` (paired-coordinate fancy indexing) with wanting the 2x2 sub-rectangle at rows 0-1, columns 0-1 (which would instead be `x[0:2, 0:2]` or `x[[0,1]][:, [0,1]]`).

### Interview Insight
**Q: Why does modifying `x[0]` change the original tensor `x`, but modifying `x[mask]` (boolean indexing) does not?**
A: Basic indexing/slicing (integer indices, slice objects, ellipsis) can always be expressed as a view into the original tensor's existing memory using offset and stride metadata — no data copy is needed, so the "view" and the "original" are genuinely the same underlying storage, and modifying one modifies the other. Boolean masking (and general fancy/integer-array indexing) selects elements that aren't necessarily contiguous or regularly strided, so PyTorch cannot represent the result as a simple view — it must allocate new memory and copy the selected values, producing a tensor that's fully independent of the original.

### Exercise
1. Create a `(5, 5)` tensor of `torch.arange(25).reshape(5,5)`. Extract the middle 3x3 sub-block using slicing.
2. Extract all values greater than 15 using boolean masking, then confirm modifying the result does NOT affect the original.
3. Use paired fancy indexing to extract the diagonal elements (0,0), (1,1), (2,2), (3,3), (4,4) in one indexing operation.

### Quiz (Chapter 6)
1. Which types of indexing return a view (shared memory), and which return a copy?
2. What does `.clone()` do, and when should you use it defensively?
3. What does `x[row_idx, col_idx]` (two equal-length index tensors) actually select — a rectangle, or paired coordinates?
4. Why is repeatedly calling `.item()` inside a training loop a performance concern?


---
# CHAPTER 7 — RESHAPING

## 1. Concept
This is one of the most operationally important chapters in the entire course — nearly every shape-mismatch bug (the single most common category of PyTorch error) traces back to a misunderstanding of `reshape`/`view`/`flatten`/`squeeze`/`unsqueeze`/`transpose`/`permute`, and the memory-layout subtlety (`contiguous()`) underlying several of them.

## 2. `reshape()` vs. `view()` — the Critical Distinction


In [ ]:
import torch

x = torch.arange(12)   # shape (12,)

x_view = x.view(3, 4)         # requires the tensor to already be "contiguous" in memory
x_reshape = x.reshape(3, 4)   # works EVEN IF the tensor isn't contiguous (copies if it must)

print("x_view:\n", x_view)
print("\nx_reshape:\n", x_reshape)
print("\nBoth produce identical RESULTS here -- the difference is about WHEN each can succeed.")

# Demonstrating the difference: view() FAILS on a non-contiguous tensor, reshape() doesn't
y = torch.arange(12).reshape(3, 4)
y_transposed = y.t()   # transpose -- this makes the tensor NON-contiguous (Section 4 explains why)
print("\ny_transposed.is_contiguous():", y_transposed.is_contiguous())

try:
    y_transposed.view(2, 6)
except RuntimeError as e:
    print("view() FAILS on non-contiguous tensor:", str(e)[:150])

# reshape() succeeds regardless -- it transparently copies when a view isn't possible
print("\nreshape() succeeds:\n", y_transposed.reshape(2, 6))


x_view:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

x_reshape:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

Both produce identical RESULTS here -- the difference is about WHEN each can succeed.

y_transposed.is_contiguous(): False
view() FAILS on non-contiguous tensor: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) inst

reshape() succeeds:
 tensor([[ 0,  4,  8,  1,  5,  9],
        [ 2,  6, 10,  3,  7, 11]])


### Line-by-Line Explanation
- `.view()` **requires** the tensor to be contiguous (its elements laid out in memory in the standard row-major order matching its logical shape) — if not, it raises a `RuntimeError` rather than silently copying, because `view()`'s entire contract is "return a new view with zero copying."
- `.reshape()` is more forgiving: it returns a view when possible (identical behavior/cost to `.view()`), but transparently falls back to copying the data into a new, contiguous tensor when a view genuinely isn't possible — this is why `.reshape()` is generally the **safer default** to reach for, while `.view()` is preferred specifically when you want to be certain no copy is happening (a performance-sensitive context) and are willing to handle the error if that assumption turns out to be wrong.

## 3. `flatten()`, `squeeze()`, `unsqueeze()`


In [ ]:
x = torch.randn(2, 3, 4)
print("Original shape:", x.shape)

flat = x.flatten()
print("flatten() -> shape:", flat.shape, " (all dims collapsed into 1)")

flat_partial = x.flatten(start_dim=1)   # flatten only from dim 1 onward -- KEEPS the batch dimension intact
print("flatten(start_dim=1) -> shape:", flat_partial.shape, " (batch dim 0 preserved, dims 1-2 merged)")

# squeeze/unsqueeze -- adding/removing size-1 dimensions
y = torch.randn(1, 3, 1, 5)
print("\ny.shape:", y.shape)
print("y.squeeze().shape:", y.squeeze().shape, " (ALL size-1 dims removed)")
print("y.squeeze(0).shape:", y.squeeze(0).shape, " (only dim 0 removed, since it's size 1)")
print("y.squeeze(1).shape:", y.squeeze(1).shape, " (dim 1 has size 3, NOT removed -- squeeze on a non-1 dim is a no-op)")

z = torch.randn(5)
print("\nz.shape:", z.shape)
print("z.unsqueeze(0).shape:", z.unsqueeze(0).shape, " (add a new dim at position 0 -- e.g., turning a single sample into a 'batch of 1')")
print("z.unsqueeze(1).shape:", z.unsqueeze(1).shape, " (add a new dim at position 1 instead)")


Original shape: torch.Size([2, 3, 4])
flatten() -> shape: torch.Size([24])  (all dims collapsed into 1)
flatten(start_dim=1) -> shape: torch.Size([2, 12])  (batch dim 0 preserved, dims 1-2 merged)

y.shape: torch.Size([1, 3, 1, 5])
y.squeeze().shape: torch.Size([3, 5])  (ALL size-1 dims removed)
y.squeeze(0).shape: torch.Size([3, 1, 5])  (only dim 0 removed, since it's size 1)
y.squeeze(1).shape: torch.Size([1, 3, 1, 5])  (dim 1 has size 3, NOT removed -- squeeze on a non-1 dim is a no-op)

z.shape: torch.Size([5])
z.unsqueeze(0).shape: torch.Size([1, 5])  (add a new dim at position 0 -- e.g., turning a single sample into a 'batch of 1')
z.unsqueeze(1).shape: torch.Size([5, 1])  (add a new dim at position 1 instead)


### Line-by-Line Explanation
- `x.flatten(start_dim=1)` — the single most common `flatten` pattern in real model code: given a `(batch, channels, height, width)` tensor of images (or any multi-dimensional per-sample data), you almost always want to flatten everything **except the batch dimension** before feeding into a fully-connected layer, since `nn.Linear` expects `(batch, features)`.
- `squeeze()` with no argument removes **every** size-1 dimension; `squeeze(dim)` removes only that specific dimension **if** it has size 1 (otherwise it's a harmless no-op, not an error) — worth knowing this doesn't raise an error on a non-1 dimension, since that surprises people expecting strict validation.
- `unsqueeze(dim)` is the exact inverse — inserts a new size-1 dimension at the specified position. **The single most common use**: converting one unbatched sample (shape `(features,)`) into a "batch of 1" (shape `(1, features)`) before feeding it to a model that always expects a batch dimension — genuinely one of the most frequently-written lines in inference code (Chapter 71).

## 4. `transpose()` and `permute()` — and Why They Break Contiguity


In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
print("Original shape:", x.shape)

transposed = x.transpose(0, 1)   # swap exactly TWO specified dimensions
print("transpose(0, 1) shape:", transposed.shape)

permuted = x.permute(2, 0, 1)    # REORDER ALL dimensions arbitrarily, in one call
print("permute(2, 0, 1) shape:", permuted.shape, " (dim 2 becomes dim 0, dim 0 becomes dim 1, dim 1 becomes dim 2)")

print("\nIs the original contiguous?", x.is_contiguous())
print("Is the permuted tensor contiguous?", permuted.is_contiguous())

# Fixing non-contiguity when you genuinely need view() or certain other operations after permute()
permuted_contiguous = permuted.contiguous()
print("After .contiguous():", permuted_contiguous.is_contiguous())


Original shape: torch.Size([2, 3, 4])
transpose(0, 1) shape: torch.Size([3, 2, 4])
permute(2, 0, 1) shape: torch.Size([4, 2, 3])  (dim 2 becomes dim 0, dim 0 becomes dim 1, dim 1 becomes dim 2)

Is the original contiguous? True
Is the permuted tensor contiguous? False
After .contiguous(): True


### Line-by-Line Explanation
- `transpose(dim0, dim1)` swaps exactly two named dimensions; `permute(*dims)` lets you specify a completely new order for **all** dimensions in one call — `permute` is the more general tool, and `transpose` is really just a special case of it.
- **Why these break contiguity**: both operations change the *logical* order of dimensions **without physically moving any data in memory** — they just change the tensor's stride metadata (how many memory positions to skip to move one step along each logical dimension). This is what makes them extremely cheap (no data copy), but it also means the tensor's logical shape no longer matches its physical memory layout in the standard row-major way — hence "non-contiguous."
- `.contiguous()` forces an actual physical reordering of the data in memory to match the tensor's current logical shape, returning a new tensor that IS contiguous — necessary before calling `.view()` (which requires contiguity) on a transposed/permuted tensor, and genuinely a very common idiom: **`.permute(...).contiguous()`** appears constantly in real model code, especially in attention mechanisms and CNN-to-RNN transition code.

### Visual Shape Reference — a Genuinely Common CNN → RNN/Linear Transition
```
Image batch:  (batch, channels, height, width)     e.g. (32, 3, 224, 224)
     ↓ Conv layers preserve this 4D structure
     ↓ .flatten(start_dim=1)  -- collapse channels/height/width into one feature axis
Flattened:    (batch, channels*height*width)         (32, 150528)
     ↓ nn.Linear expects exactly this 2D (batch, features) shape
```

### Common Mistakes
- Calling `.view()` on a tensor right after `.transpose()`/`.permute()` without `.contiguous()` first — raises a `RuntimeError`; either use `.reshape()` instead (which handles this automatically) or explicitly call `.contiguous()` before `.view()`.
- Confusing `flatten()` (no arguments, collapses everything including the batch dimension) with the far more commonly-needed `flatten(start_dim=1)` (preserves the batch dimension) — a genuine, easy mistake that silently corrupts batch structure.
- Assuming `squeeze(dim)` raises an error on a non-size-1 dimension — it doesn't; it's a silent no-op, which can mask a bug where you expected a dimension to already be size 1 but it wasn't.
- Forgetting `unsqueeze()` when feeding a single unbatched sample to a model — resulting in a shape-mismatch error deep inside the first layer, often confusing to trace back to its true root cause.

### Best Practices
- Default to `.reshape()` over `.view()` unless you specifically need the "never silently copies" guarantee for performance-critical code.
- Always use `flatten(start_dim=1)`, not bare `flatten()`, when preparing batched data for a `Linear` layer.
- Get comfortable with the `.permute(...).contiguous()` idiom — it appears constantly in attention/CNN code.

### Interview Insight
**Q: You call `.view()` on a tensor right after `.permute()` and get `RuntimeError: view size is not compatible with input tensor's size and stride`. What's happening, and what are your two options to fix it?**
A: `.permute()` reorders a tensor's logical dimensions purely by changing its stride metadata, without physically moving data in memory — the resulting tensor's logical shape no longer matches a standard contiguous row-major memory layout. `.view()` requires contiguity because its entire contract is "produce a new view with zero data copying," which isn't achievable for this specific reshape on this non-contiguous tensor. The two fixes: call `.contiguous()` before `.view()` to force a physical memory reorder first (making the tensor contiguous again), or simply use `.reshape()` instead of `.view()`, which handles this automatically by copying only when genuinely necessary.

### Exercise
1. Create a `(4, 3, 2)` tensor. Use `permute` to reorder it to `(2, 4, 3)`. Check `.is_contiguous()` before and after.
2. Take a `(10,)` vector and `unsqueeze` it to shape `(1, 10)`, then to `(1, 1, 10)`.
3. Create a `(32, 3, 224, 224)` tensor (simulating an image batch) and flatten it correctly for a `Linear` layer, preserving the batch dimension.

### Quiz (Chapter 7)
1. What's the precise difference between `.view()` and `.reshape()`, in terms of when each can succeed?
2. Why do `transpose()`/`permute()` make a tensor non-contiguous, given that they don't copy any data?
3. What does `flatten(start_dim=1)` preserve that bare `flatten()` does not?
4. What's the single most common real-world use of `unsqueeze()`?


---
# CHAPTER 8 — BROADCASTING

## 1. Concept
Broadcasting lets PyTorch perform elementwise operations between tensors of **different but compatible** shapes, without you needing to manually expand either tensor to match the other — it directly extends NumPy's broadcasting rules identically.

## 2. The Broadcasting Rule, Precisely
Two tensors' shapes are compatible for broadcasting if, comparing dimensions from the **right** (trailing dimension) toward the left:
1. The dimensions are equal, **or**
2. One of the dimensions is **1**, **or**
3. One of the tensors simply doesn't have that dimension (treated as if it had size 1).

Any dimension that's `1` (or missing) gets conceptually "stretched" to match the other tensor's size for that dimension — with no actual memory duplication; PyTorch computes the broadcasted result directly and efficiently.


In [ ]:
import torch

# Example 1: batch of feature vectors, each normalized by the SAME per-feature scale
batch = torch.randn(32, 10)      # 32 samples, 10 features each
scale = torch.randn(10)          # ONE scale value per feature -- shape (10,)

result = batch * scale           # broadcasts (10,) against (32, 10) -- applies scale to EVERY sample
print("batch shape:", batch.shape, " scale shape:", scale.shape, " result shape:", result.shape)

# Example 2: adding a per-sample bias, one value per row
bias_per_sample = torch.randn(32, 1)   # shape (32, 1) -- note the explicit trailing 1
result2 = batch + bias_per_sample       # broadcasts (32, 1) against (32, 10)
print("result2 shape:", result2.shape)

# Example 3: a genuinely common CNN pattern -- normalizing a batch of images by per-channel stats
images = torch.randn(16, 3, 224, 224)          # (batch, channels, height, width)
channel_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)   # reshape for broadcasting compatibility
normalized = images - channel_mean
print("\nimages shape:", images.shape, " channel_mean shape:", channel_mean.shape, " normalized shape:", normalized.shape)


batch shape: torch.Size([32, 10])  scale shape: torch.Size([10])  result shape: torch.Size([32, 10])
result2 shape: torch.Size([32, 10])

images shape: torch.Size([16, 3, 224, 224])  channel_mean shape: torch.Size([1, 3, 1, 1])  normalized shape: torch.Size([16, 3, 224, 224])


### Line-by-Line Explanation
- `batch * scale` — `batch` is `(32, 10)`, `scale` is `(10,)`. Comparing from the right: the trailing dimension `10` matches `10` — fine. `scale` simply has no dimension corresponding to `batch`'s leading `32` — treated as size 1, broadcast (stretched) to `32`. Result: `(32, 10)`, with `scale` applied identically to every one of the 32 samples.
- `channel_mean.view(1, 3, 1, 1)` — this explicit reshape is **essential** and a very common real pattern: a raw per-channel mean tensor of shape `(3,)` would broadcast **incorrectly** against `(16, 3, 224, 224)` (the rule would try to align `3` against the trailing `224`, which don't match, raising an error) — reshaping to `(1, 3, 1, 1)` explicitly places the `3` in the **channel** position, letting broadcasting correctly stretch dimensions 0, 2, 3 (batch, height, width) while matching dimension 1 (channels) exactly.

## 3. A Broadcasting Failure, Debugged


In [ ]:
a = torch.randn(4, 3)
b = torch.randn(4)

try:
    result = a + b
except RuntimeError as e:
    print("Broadcasting FAILS:", str(e)[:200])

print("\nWhy: comparing from the right -- a's trailing dim is 3, b's trailing (only) dim is 4.")
print("3 != 4, and NEITHER is 1 -- broadcasting rule violated, no valid alignment exists.")

# The FIX: make the intent explicit with unsqueeze -- if you meant 'b' to be one value per ROW
b_fixed = b.unsqueeze(1)   # reshape (4,) -> (4, 1)
result_fixed = a + b_fixed
print("\nAfter b.unsqueeze(1) -> shape", b_fixed.shape, ": now broadcasts correctly, result shape:", result_fixed.shape)


Broadcasting FAILS: The size of tensor a (3) must match the size of tensor b (4) at non-singleton dimension 1

Why: comparing from the right -- a's trailing dim is 3, b's trailing (only) dim is 4.
3 != 4, and NEITHER is 1 -- broadcasting rule violated, no valid alignment exists.

After b.unsqueeze(1) -> shape torch.Size([4, 1]) : now broadcasts correctly, result shape: torch.Size([4, 3])


### Internal Working
Broadcasting is implemented via **stride manipulation**, not actual memory duplication — when a dimension is "stretched," PyTorch sets that dimension's stride to `0`, meaning "reading any index along this dimension returns the same underlying memory location." This is why broadcasting is essentially free in both memory and (for most operations) computation — genuinely no wasted work, just clever indexing metadata.

### `torch.broadcast_shapes()` — Predicting the Result Shape Without Running the Operation


In [ ]:
shape_a = (32, 1, 10)
shape_b = (1, 5, 10)
predicted_shape = torch.broadcast_shapes(shape_a, shape_b)
print("Predicted broadcast result shape:", predicted_shape)
print("\nUseful for VERIFYING your shape reasoning BEFORE running an expensive operation,")
print("especially when debugging a complex multi-step tensor pipeline.")


Predicted broadcast result shape: torch.Size([32, 5, 10])

Useful for VERIFYING your shape reasoning BEFORE running an expensive operation,
especially when debugging a complex multi-step tensor pipeline.


### Common Mistakes
- Assuming any two tensors will broadcast "somehow" without checking the trailing-dimension alignment rule explicitly — the most common source of a `RuntimeError: The size of tensor a (X) must match the size of tensor b (Y) at non-singleton dimension Z`.
- Forgetting to reshape a per-channel/per-feature tensor into the correct broadcasting position (as the `channel_mean.view(1, 3, 1, 1)` example demonstrated) — silently broadcasting against the *wrong* dimension if the shapes happen to accidentally align in an unintended way (a much more dangerous failure than an explicit error, since it produces a wrong-but-plausible-looking result with no crash).
- Not realizing broadcasting can silently succeed with an unintended interpretation when two unrelated dimensions happen to have the same size — always double check the *intended* alignment, not just whether an operation runs without error.

### Interview Insight
**Q: You want to add a per-sample bias vector of shape `(32,)` to a batch of feature vectors of shape `(32, 10)`, but PyTorch raises a broadcasting error. Walk through why, and the fix.**
A: Broadcasting compares shapes from the trailing dimension leftward: `(32, 10)`'s trailing dim is `10`; `(32,)`'s only dim is `32`. `10 != 32` and neither is `1`, so no valid broadcast alignment exists — PyTorch has no way to know you meant the `32` in the bias to align with the *leading* (batch) dimension of the feature tensor, not the trailing one. The fix is `bias.unsqueeze(1)`, reshaping to `(32, 1)` — now the trailing dimensions align as `1` vs `10` (broadcasts correctly, stretching the bias across all 10 features) and the leading dimensions align as `32` vs `32` (match exactly) — producing the intended per-sample bias addition.

### Exercise
1. Create a `(5, 1, 4)` tensor and a `(1, 3, 4)` tensor. Predict the broadcast result shape using `torch.broadcast_shapes()`, then confirm by actually adding them.
2. Deliberately create two tensors that FAIL to broadcast, reproduce the error, then fix it with `unsqueeze`.

### Quiz (Chapter 8)
1. State the broadcasting compatibility rule precisely, in your own words.
2. Why is broadcasting essentially free in memory, despite conceptually "stretching" a dimension?
3. Why is `channel_mean.view(1, 3, 1, 1)` necessary rather than just using the raw `(3,)` shape when normalizing an image batch?
4. What's more dangerous: a broadcasting operation that raises an error, or one that silently succeeds with an unintended alignment? Why?


---
# CHAPTER 9 — MATHEMATICAL OPERATIONS

## 1. Concept
PyTorch provides the full suite of elementwise math (identical in spirit to NumPy's ufuncs) plus **reduction** operations (`sum`, `mean`, `max`, ...) that collapse one or more dimensions — and the `dim`/`keepdim` parameters governing exactly *which* dimension(s) a reduction collapses are worth mastering deeply, since getting them wrong is a frequent, subtle bug source.

## 2. Elementwise Operations


In [ ]:
import torch

x = torch.tensor([1.0, 4.0, 9.0, 16.0])

print("x + 5:      ", x + 5)
print("x * 2:      ", x * 2)
print("x ** 0.5:   ", x ** 0.5)          # equivalent to torch.sqrt(x)
print("torch.exp(x):", torch.exp(x))
print("torch.log(x):", torch.log(x))
print("torch.abs(torch.tensor([-3.0, 2.0])):", torch.abs(torch.tensor([-3.0, 2.0])))
print("torch.clamp(x, min=2, max=10):", torch.clamp(x, min=2, max=10))


x + 5:       tensor([ 6.,  9., 14., 21.])
x * 2:       tensor([ 2.,  8., 18., 32.])
x ** 0.5:    tensor([1., 2., 3., 4.])
torch.exp(x): tensor([2.7183e+00, 5.4598e+01, 8.1031e+03, 8.8861e+06])
torch.log(x): tensor([0.0000, 1.3863, 2.1972, 2.7726])
torch.abs(torch.tensor([-3.0, 2.0])): tensor([3., 2.])
torch.clamp(x, min=2, max=10): tensor([ 2.,  4.,  9., 10.])


### `torch.clamp()` — a Genuinely Important Function in Real Model Code
`clamp(x, min=, max=)` restricts every element to a specified range — used constantly for **gradient clipping** (Chapter 77), keeping probabilities/log-values within a numerically safe range before a `log()` call (avoiding `log(0) = -inf`), and implementing certain activation-like functions manually.

## 3. Reduction Operations — `dim` and `keepdim`, Precisely


In [ ]:
x = torch.arange(24).reshape(2, 3, 4).float()   # shape (2, 3, 4)
print("x.shape:", x.shape)

total = x.sum()                    # NO dim -- reduces ALL dimensions, returns a 0-d scalar tensor
print("\nx.sum() (all dims):", total.item(), " shape:", total.shape)

sum_dim0 = x.sum(dim=0)            # collapses dim 0 (size 2) -- result shape (3, 4)
print("x.sum(dim=0) shape:", sum_dim0.shape)

sum_dim1 = x.sum(dim=1)            # collapses dim 1 (size 3) -- result shape (2, 4)
print("x.sum(dim=1) shape:", sum_dim1.shape)

sum_dim_neg1 = x.sum(dim=-1)       # collapses the LAST dim (size 4) -- result shape (2, 3)
print("x.sum(dim=-1) shape:", sum_dim_neg1.shape)

sum_keepdim = x.sum(dim=1, keepdim=True)    # collapses dim 1 but KEEPS it as size 1, instead of removing it
print("x.sum(dim=1, keepdim=True) shape:", sum_keepdim.shape, " (still 3D, dim 1 is now size 1 instead of gone)")


x.shape: torch.Size([2, 3, 4])

x.sum() (all dims): 276.0  shape: torch.Size([])
x.sum(dim=0) shape: torch.Size([3, 4])
x.sum(dim=1) shape: torch.Size([2, 4])
x.sum(dim=-1) shape: torch.Size([2, 3])
x.sum(dim=1, keepdim=True) shape: torch.Size([2, 1, 4])  (still 3D, dim 1 is now size 1 instead of gone)


### Line-by-Line Explanation — the Precise Mental Model for `dim`
**The `dim` argument specifies which dimension gets *collapsed/removed* (by default) — NOT which dimension is preserved.** This is the single most common source of confusion: `x.sum(dim=0)` on a `(2, 3, 4)` tensor means "collapse dimension 0 (the one of size 2) by summing across it," leaving `(3, 4)` — the *other* dimensions survive.

- `keepdim=True` — instead of *removing* the reduced dimension entirely, keeps it as a size-1 dimension. This is **essential** whenever you need the reduction's result to broadcast correctly back against the original tensor (Chapter 8) — e.g., normalizing `x` by subtracting its own per-row mean: `x - x.mean(dim=1, keepdim=True)` broadcasts correctly (shape `(2,3,4)` minus `(2,1,4)`... wait, more precisely for a 2D case: `(N, D) - (N, 1)` broadcasts correctly); without `keepdim=True`, `x.mean(dim=1)` would have shape `(2, 4)` and the subtraction would either fail outright or (worse) broadcast against the *wrong* dimension.

## 4. `argmax`/`argmin` — Finding the Index, Not the Value


In [ ]:
logits = torch.tensor([[2.1, 0.5, 3.8, 1.2], [0.9, 4.2, 1.1, 0.3]])   # shape (2, 4) -- 2 samples, 4 classes

predicted_classes = logits.argmax(dim=1)    # for EACH sample (dim=0 is preserved), find the index of the max VALUE along dim=1
print("logits shape:", logits.shape)
print("predicted classes (argmax over dim=1):", predicted_classes)
print("\nThis is EXACTLY how you convert raw classification logits into predicted class labels --")
print("dim=1 here means 'reduce over the CLASS dimension,' leaving one predicted class PER SAMPLE.")


logits shape: torch.Size([2, 4])
predicted classes (argmax over dim=1): tensor([2, 1])

This is EXACTLY how you convert raw classification logits into predicted class labels --
dim=1 here means 'reduce over the CLASS dimension,' leaving one predicted class PER SAMPLE.


### Common Mistakes
- Confusing which dimension `dim=` refers to — remembering "dim specifies what gets collapsed" (not what survives) takes deliberate practice; always sanity-check by predicting the output shape before running the code.
- Forgetting `keepdim=True` when a reduction's result needs to broadcast back against the original tensor — leads to either a broadcasting error or, worse, a silently wrong broadcast against an unintended dimension.
- Using `argmax(dim=0)` when you meant `dim=1` (or vice versa) for a `(batch, num_classes)` logits tensor — a genuinely easy off-by-one-dimension mistake that silently produces syntactically valid but semantically wrong "predicted classes."

### Interview Insight
**Q: For a `(batch_size, num_classes)` logits tensor, why is `logits.argmax(dim=1)` correct for getting predicted class labels, and what would `dim=0` give you instead?**
A: `dim=1` is the class dimension — `argmax(dim=1)` finds, for each sample (each row, dimension 0 is preserved), the index along the class dimension with the highest logit value, producing one predicted class per sample — exactly what's needed. `argmax(dim=0)` would instead collapse the *batch* dimension, finding for each class (each column) which sample in the batch had the highest logit for that class — a fundamentally different and essentially never-useful computation in this context. Getting `dim` backward here produces code that runs without error but computes something semantically meaningless.

### Exercise
1. Create a `(3, 4)` tensor. Compute its sum along `dim=0` and along `dim=1` separately, and verify the resulting shapes match your prediction before running the code.
2. Given a `(10, 5)` "features" tensor, normalize each row to have zero mean using `keepdim=True` correctly.
3. Given a `(16, 10)` logits tensor (16 samples, 10 classes), compute the predicted class for every sample using `argmax`.

### Quiz (Chapter 9)
1. Does `dim=` in a reduction specify the dimension that's collapsed, or the dimension that survives?
2. What does `keepdim=True` change, and when is it essential?
3. For a `(batch, num_classes)` logits tensor, which `dim` gives you per-sample predicted classes via `argmax`?
4. What is `torch.clamp()` commonly used for in real training code?


---
# CHAPTER 10 — MATRIX OPERATIONS

## 1. Concept
Matrix multiplication is the single most computationally central operation in deep learning — every `nn.Linear` layer, every attention mechanism, is fundamentally matrix multiplication. PyTorch provides several related functions, each suited to a specific dimensionality of input.

## 2. `torch.mm()`, `torch.matmul()`, `torch.bmm()` — Choosing the Right One
| Function | Input dims | Use case |
|---|---|---|
| `torch.mm(a, b)` | Both strictly 2D | Plain matrix multiplication — the simplest, most restrictive case |
| `torch.bmm(a, b)` | Both strictly 3D, `(batch, n, m)` | **Batched** matrix multiplication — the SAME operation applied independently across a batch dimension |
| `torch.matmul(a, b)` (or the `@` operator) | Flexible — 1D, 2D, or higher, with broadcasting | The **general-purpose** function; handles every case `mm`/`bmm` handle, plus more, via broadcasting rules over any leading "batch" dimensions |


In [ ]:
import torch

# Plain 2D matrix multiplication
a = torch.randn(3, 4)
b = torch.randn(4, 5)
result_mm = torch.mm(a, b)
result_matmul = a @ b   # the @ operator is shorthand for torch.matmul()
print("mm result shape:", result_mm.shape, "  matmul (@) result shape:", result_matmul.shape)

# Batched matrix multiplication -- the standard shape for a batch of independent matrix pairs
batch_a = torch.randn(32, 3, 4)   # 32 independent 3x4 matrices
batch_b = torch.randn(32, 4, 5)   # 32 independent 4x5 matrices
result_bmm = torch.bmm(batch_a, batch_b)
print("\nbmm result shape:", result_bmm.shape, " (32 independent 3x5 results, one per batch element)")

# matmul() handles the SAME batched case automatically -- and MORE (broadcasting across batch dims)
result_matmul_batched = batch_a @ batch_b
print("matmul on the same batched inputs:", result_matmul_batched.shape, " (identical result)")


mm result shape: torch.Size([3, 5])   matmul (@) result shape: torch.Size([3, 5])

bmm result shape: torch.Size([32, 3, 5])  (32 independent 3x5 results, one per batch element)
matmul on the same batched inputs: torch.Size([32, 3, 5])  (identical result)


### Line-by-Line Explanation
- `torch.bmm()` requires **exactly** 3D inputs with matching batch sizes — it's the strict, explicit tool.
- `torch.matmul()` (and the `@` operator, its idiomatic shorthand) is strictly more general: for 2D inputs it behaves like `mm`; for higher-dimensional inputs, it treats all leading dimensions as batch dimensions and broadcasts them (Chapter 8's rules) if they don't match exactly — e.g., a `(32, 3, 4)` and a single shared `(4, 5)` matrix (not batched) will correctly broadcast, applying the same `(4,5)` matrix to every one of the 32 batch elements, something `bmm` cannot do at all (it demands matching batch dimensions on both sides).
- **In practice, `torch.matmul()`/`@` is what you'll use nearly always** — it's strictly more capable and equally efficient; `mm`/`bmm` exist mainly for explicitness/clarity in specific contexts or older code.

## 3. `torch.einsum()` — the Advanced, General-Purpose Tool


In [ ]:
# einsum expresses ANY tensor contraction using Einstein summation notation --
# extremely powerful, genuinely common in attention/transformer implementations

a = torch.randn(32, 8, 64)   # (batch, heads, dim) -- e.g. query vectors in multi-head attention
b = torch.randn(32, 8, 64)   # (batch, heads, dim) -- e.g. key vectors

# Computing a batched, per-head dot product between corresponding query/key vectors:
dot_products = torch.einsum('bhd,bhd->bh', a, b)
print("einsum result shape:", dot_products.shape, " (one scalar dot-product per batch element, per head)")

# The equivalent WITHOUT einsum -- more verbose, arguably less readable once you know einsum notation:
dot_products_manual = (a * b).sum(dim=-1)
print("Manual equivalent shape:", dot_products_manual.shape)
print("Identical result:", torch.allclose(dot_products, dot_products_manual))

# A genuinely common attention-style pattern: query-key SIMILARITY MATRIX across ALL pairs
queries = torch.randn(32, 10, 64)    # (batch, seq_len_q, dim)
keys = torch.randn(32, 20, 64)       # (batch, seq_len_k, dim)
attention_scores = torch.einsum('bqd,bkd->bqk', queries, keys)
print("\nAttention scores shape:", attention_scores.shape, " (batch, query_positions, key_positions)")


einsum result shape: torch.Size([32, 8])  (one scalar dot-product per batch element, per head)
Manual equivalent shape: torch.Size([32, 8])
Identical result: False

Attention scores shape: torch.Size([32, 10, 20])  (batch, query_positions, key_positions)


### Line-by-Line Explanation — Reading Einstein Summation Notation
- `'bhd,bhd->bh'` — read as: input 1 has dimensions labeled `(b, h, d)`, input 2 has dimensions labeled `(b, h, d)` too, and the output should have dimensions `(b, h)`. Any label that appears in the inputs but **not** in the output (here, `d`) is **summed over** — this is exactly a dot product along the `d` (feature) dimension, computed independently for every `(b, h)` combination.
- `'bqd,bkd->bqk'` — inputs have dimensions `(b,q,d)` and `(b,k,d)`; output is `(b,q,k)`; `d` disappears (summed over) while `q` and `k` both survive **separately** (not summed together) — this computes every pairwise dot product between each of the `q` query positions and each of the `k` key positions, for every batch element — literally the core computation inside scaled dot-product attention (Chapter 49).
- `einsum` is genuinely more **readable once fluent in the notation** than the equivalent chain of `unsqueeze`/`transpose`/`matmul`/`sum` calls it replaces, and is extremely common in real Transformer/attention implementation code — worth the investment to become comfortable with, even though it has a real learning curve at first.

### Common Mistakes
- Using `torch.mm()` on batched (3D+) inputs — raises a dimensionality error immediately; `mm` is strictly 2D-only.
- Using `torch.bmm()` when one input isn't actually batched (a single shared matrix) — `bmm` requires both inputs to have matching batch dimensions; `matmul`'s broadcasting handles this mixed case, `bmm` does not.
- Getting an einsum subscript string wrong (e.g., accidentally repeating a label that should be distinct, or vice versa) — produces a tensor of an unexpected shape or an outright wrong numerical result, often silently (no error) if the string still describes a *valid* (just unintended) contraction; always verify the output shape matches your expectation.

### Interview Insight
**Q: What's the practical difference between `torch.matmul()` and `torch.bmm()`, and why is `matmul()` generally the better default?**
A: `torch.bmm()` strictly requires both inputs to be exactly 3D with matching batch dimensions on both sides. `torch.matmul()` handles 2D inputs (behaving like `mm`), exactly-matching 3D+ batched inputs (behaving like `bmm`), AND mixed cases where one input has a batch dimension and the other doesn't, via broadcasting the non-batched input across the batch — a genuinely common real situation (e.g., applying the same shared weight matrix to every element of a batch) that `bmm` cannot express at all. Since `matmul()` is a strict superset of what `mm`/`bmm` can do, with no performance penalty, it's the practical default; `mm`/`bmm` are mainly useful for their explicit dimensionality requirements acting as a form of self-documentation/validation in code where you want to guarantee a specific input shape.

### Exercise
1. Multiply a `(10, 20)` matrix by a `(20, 5)` matrix using both `torch.mm()` and `@`, confirming identical results.
2. Given a batch of 16 independent `(3, 3)` matrices and a SINGLE shared `(3, 3)` matrix, use `torch.matmul()` to multiply each batch element by the shared matrix (this would fail with `bmm`).
3. Rewrite `torch.einsum('bqd,bkd->bqk', queries, keys)` using explicit `transpose`/`matmul` calls instead, and verify the results match.

### Quiz (Chapter 10)
1. What's the dimensionality requirement for `torch.mm()` vs `torch.bmm()` vs `torch.matmul()`?
2. Why can `torch.matmul()` handle a case that `torch.bmm()` cannot?
3. In einsum notation `'bhd,bhd->bh'`, why does the `d` dimension disappear from the output?
4. Why is `torch.einsum` particularly common in attention/Transformer implementations?

### Chapter 10 Summary
- `torch.matmul()`/`@` is the general-purpose, nearly-always-correct choice for matrix multiplication, handling 2D, batched, and mixed-batching cases via broadcasting.
- `torch.einsum()` expresses arbitrary tensor contractions via Einstein notation — genuinely essential vocabulary for reading and writing attention mechanism code.

---

## END OF NOTEBOOK 2 — What You Should Now Be Able To Do
- Index and slice tensors fluently, correctly distinguishing which operations return views versus copies, and use `.clone()` deliberately.
- Reshape tensors confidently using `reshape`/`view`/`flatten`/`squeeze`/`unsqueeze`/`transpose`/`permute`, and diagnose and fix contiguity errors.
- Reason correctly about broadcasting compatibility, predict result shapes before running code, and debug broadcasting failures.
- Use `dim`/`keepdim` correctly in reduction operations, and never again confuse which dimension gets collapsed.
- Choose the correct matrix multiplication function (`matmul` as the general default) and read basic `einsum` notation.

## Before Moving to Notebook 3 (Chapters 11–15: Concatenation and Stacking, Randomness and Reproducibility, Autograd, Backpropagation, Gradient Management)
 
